##### Prep model data

This script cleans the raw outputs of the ABM and prepares them for use in the figure scripts. 
It creates a sub-folder to store the cleaned results data in this repo, but that is ignored in .gitignore. 
\
\
To run the script, specify the scenarios you want to run in 'scenarios' dictionary at the top. You can then run the entire script.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# get filepath 
cd = Path.cwd() 

# create folder to store clean data 
out_dir = Path(f"{cd}/Clean_Data")
out_dir.mkdir(parents=True, exist_ok=True)

In [3]:
##### ---------- MANUAL SET UP ----------

# for each sceanrio, set scenario name and run date for each watershed
scenarios = [
    {
        'scenario': 'calibration',
        'run_dates': {
            'GBSJ': '2026-07-16',
            'LCRR': '2026-07-16',
            'MILK': '2026-07-16',
        }
    }
]

In [4]:
##### ---------- Prep validation data ----------

# import and select columns
USGS_data = pd.read_csv("/netfiles/ciroh/cmanitiu/USGS/Data/USGS_Water_Use/CLEAN/total_water_use_2000_2020.csv")
USGS_data = USGS_data[['Year', 'HUC12', 'region', 'public_supply_total_mil_gal','irr_total_mil_gal', 'thermo_consumptive_million_gal']]

# replace CAN data with n/a (USGS has 0's by default)
all_regions = pd.read_csv('/netfiles/ciroh/cmanitiu/USGS/Data/Water_Basin_Boundaries/Filtered/HUC12/US_region_list.csv')
all_regions = all_regions.rename(columns={'huc12': 'HUC12'})
USGS_data = USGS_data.merge(all_regions, on='HUC12', how='left')
USGS_data.loc[USGS_data['in_us'] == 0, ['public_supply_total_mil_gal', 'irr_total_mil_gal', 'thermo_consumptive_million_gal']] = np.nan

# rename columns
USGS_data = USGS_data.rename(columns={
    'region': 'Watershed',
    'public_supply_total_mil_gal': 'public_supply_mil_gal_validation',
    'irr_total_mil_gal': 'irrigation_mil_gal_validation',
    'thermo_consumptive_million_gal': 'thermo_consumptive_million_gal_validation'
    })

USGS_data['HUC12'] = USGS_data['HUC12'].astype(str).str.zfill(12)

USGS_data_GBSJ = USGS_data[USGS_data['Watershed'] == 'GBSJ']
GBSJ_dir = Path(f"{cd}/Clean_Data/GBSJ")
GBSJ_dir.mkdir(parents=True, exist_ok=True)
USGS_data_GBSJ.to_csv(f"{cd}/Clean_Data/GBSJ/USGS_validation.csv", index=False)

USGS_data_LCRR = USGS_data[USGS_data['Watershed'] == 'LCRR']
LCRR_dir = Path(f"{cd}/Clean_Data/LCRR")
LCRR_dir.mkdir(parents=True, exist_ok=True)
USGS_data_LCRR.to_csv(f"{cd}/Clean_Data/LCRR/USGS_validation.csv", index=False)

USGS_data_MILK = USGS_data[USGS_data['Watershed'] == 'MILK']
MILK_dir = Path(f"{cd}/Clean_Data/MILK")
MILK_dir.mkdir(parents=True, exist_ok=True)
USGS_data_MILK.to_csv(f"{cd}/Clean_Data/MILK/USGS_validation.csv", index=False)

In [5]:
##### ---------- CLEAN ENSEMBLE DATA ----------

### ----- DEFINE FUNCTION -----

def clean_ensemble_runs(watershed, scenario, run_date, run_ids):

    all_runs = []

    for run_id in run_ids:

        # import raw data
        modelled = pd.read_csv(f"/netfiles/ciroh/kandrew6/Water_Use_Model/Outputs_{watershed}/{scenario}-{run_date}/wateruse_log-{run_id}.csv")

        # rename columns
        modelled.rename(columns={
            'huc12 string': 'HUC12',
            'year': 'Year',
            'projected public water use': f'public_supply_mil_gal_{scenario}',
            'projected irrigation water use': f'irrigation_mil_gal_{scenario}'
        }, inplace=True)

        col = ['Year', 'HUC12', f'public_supply_mil_gal_{scenario}', f'irrigation_mil_gal_{scenario}']
        modelled = modelled[col]
        modelled['Watershed'] = watershed
        modelled['HUC12'] = modelled['HUC12'].astype(str).str.zfill(12)
        modelled['run_ID'] = run_id

        all_runs.append(modelled)

    # combine all run_ids into one dataframe
    combined = pd.concat(all_runs, ignore_index=True)

    # save
    save_dir = Path(f"{cd}/Clean_Data/{watershed}")
    combined.to_csv(save_dir / f"{scenario}-{run_date}.csv", index=False)

    print(f'{watershed} scenario: {scenario} {run_date} saved.')

### ----- RUN -----

run_ids = [f"{i:03d}" for i in range(1, 51)]

for s in scenarios:
    scenario = s['scenario']
    for watershed, run_date in s['run_dates'].items():
        clean_ensemble_runs(watershed, scenario, run_date, run_ids)

GBSJ scenario: calibration 2026-07-16 saved.
LCRR scenario: calibration 2026-07-16 saved.
MILK scenario: calibration 2026-07-16 saved.
